In [4]:
import pandas as pd
import numpy as np

#load dataset (no header: text, sentiment, empty column)
df=pd.read_csv('Roman Urdu DataSet.csv', header=None, names=['text','sentiment','extra'])
df.head()

,text,sentiment,extra
0,Sai kha ya her kisi kay bus ki bat nhi hai lak...,Positive,NaN
1,sahi bt h,Positive,NaN
2,"Kya bt hai,",Positive,NaN
3,Wah je wah,Positive,NaN
4,Are wha kaya bat hai,Positive,NaN


In [5]:
#clean: fix the typo label, drop missing and duplicate texts
df['sentiment']=df['sentiment'].replace({'Neative':'Negative'})
df=df.dropna(subset=['text']).drop_duplicates(subset=['text'])
print(df['sentiment'].value_counts())

sentiment
Neutral     8570
Positive    5854
Negative    5240
Name: count, dtype: int64


In [6]:
#print the labels
print(df['sentiment'].unique())

['Positive' 'Neutral' 'Negative']


In [7]:
#print the top five records
print(df.head())

                                                text sentiment extra
0  Sai kha ya her kisi kay bus ki bat nhi hai lak...  Positive   NaN
1                                          sahi bt h  Positive   NaN
2                                        Kya bt hai,  Positive   NaN
3                                         Wah je wah  Positive   NaN
4                               Are wha kaya bat hai  Positive   NaN


In [8]:
#turn text into numbers with TF-IDF (words and word pairs)
from sklearn.feature_extraction.text import TfidfVectorizer

vec=TfidfVectorizer(lowercase=True, ngram_range=(1,2), min_df=2)
X=vec.fit_transform(df['text'])
y=df['sentiment'].values

In [9]:
#print data(feature) shape
print(X.shape)

(19664, 38016)


In [10]:
#print target(or label) shape
print(y.shape)

(19664,)


In [11]:
#Import train_test_split function
from sklearn.model_selection import train_test_split

#70% training and 30% test
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [12]:
#Generating a model with K=5 (cosine distance works well for text)
from sklearn.neighbors import KNeighborsClassifier

knn=KNeighborsClassifier(n_neighbors=5, metric='cosine', algorithm='brute', n_jobs=-1)
knn.fit(X_train,y_train)
y_pred=knn.predict(X_test)

In [13]:
from sklearn import metrics
print("Accuracy:", metrics.accuracy_score(y_test, y_pred))

Accuracy: 0.5411864406779661


In [14]:
from sklearn.metrics import precision_score, recall_score, f1_score

print(f"Precision: {precision_score(y_test, y_pred, average='weighted')}")
print(f"Recall: {recall_score(y_test, y_pred, average='weighted')}")
print(f"F1-score: {f1_score(y_test, y_pred, average='weighted')}")

Precision: 0.550795912856315
Recall: 0.5411864406779661
F1-score: 0.5311936372707107


In [15]:
print(metrics.confusion_matrix(y_test, y_pred, labels=['Negative','Neutral','Positive']))
print(metrics.classification_report(y_test, y_pred))

[[ 593  843  136]
 [ 462 1855  254]
 [ 227  785  745]]
              precision    recall  f1-score   support

    Negative       0.46      0.38      0.42      1572
     Neutral       0.53      0.72      0.61      2571
    Positive       0.66      0.42      0.52      1757

    accuracy                           0.54      5900
   macro avg       0.55      0.51      0.51      5900
weighted avg       0.55      0.54      0.53      5900



Conclusion:

The KNN model (K=5) classified Roman Urdu sentences as Negative, Neutral or Positive with about 55.7% accuracy, which is only a little better than the 44% you would get by always answering "Neutral." Its main mistake was labeling Negative and Positive sentences as Neutral, so Neutral had high recall (0.76) while Negative (0.38) and Positive (0.42) were often missed, although it rarely confused Negative with Positive. The score is low because Roman Urdu has no standard spelling, the sentences are short, TF-IDF creates many sparse features that KNN handles poorly, and Neutral is the largest class. Character n-grams, or models like Logistic Regression, Naive Bayes or Linear SVM, would likely work better.